# PJM Day-Ahead Forecasting on Kaggle

This notebook runs the PJM/COMED pipeline on Kaggle GPU.

- Recommended accelerator: **GPU**
- TPU is **not supported** by the current PyTorch Lightning / NeuralForecast path
- Fast mode keeps only `nbeatsx` and lowers tuning trials


In [ ]:
from pathlib import Path
import os
import shutil
import sys

REPO_ROOT = Path('/kaggle/working/pjm_remaster')
if not REPO_ROOT.exists():
    raise FileNotFoundError('Clone or copy the repo into /kaggle/working/pjm_remaster first.')

os.chdir(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print('repo', REPO_ROOT)


## Install project dependencies

Kaggle GPU images usually already include CUDA PyTorch. Avoid reinstalling `torch` unless you know you need to.

In [ ]:
!python -m pip install -e .[dev]
!python -m pip install neuralforecast optuna holidays pyarrow


In [ ]:
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))


## Build Kaggle-local config

If internet is disabled, upload `PJM.csv` as a Kaggle Dataset and point `LOCAL_CSV_PATH` to it.

In [ ]:
from pjm_forecast.kaggle import build_kaggle_config, kaggle_artifact_paths

BASE_CONFIG = REPO_ROOT / 'configs' / 'pjm_day_ahead_v1.yaml'
KAGGLE_CONFIG = REPO_ROOT / 'configs' / 'pjm_day_ahead_kaggle_runtime.yaml'
OUTPUT_ROOT = Path('/kaggle/working/pjm_remaster_run')
LOCAL_CSV_PATH = None  # e.g. '/kaggle/input/pjm-csv/PJM.csv'

config_path = build_kaggle_config(
    base_config_path=BASE_CONFIG,
    output_config_path=KAGGLE_CONFIG,
    output_root=OUTPUT_ROOT,
    local_csv_path=LOCAL_CSV_PATH,
    fast_mode=True,
    include_retrieval=True,
)
print(config_path)
print(kaggle_artifact_paths(OUTPUT_ROOT))


## Run stage by stage

You can stop after any stage and resume later.

In [ ]:
from pjm_forecast.pipeline import run_pipeline

run_pipeline(str(config_path), split='validation', start_from='prepare_data', stop_after='prepare_data')


In [ ]:
run_pipeline(str(config_path), split='validation', start_from='tune_nbeatsx', stop_after='tune_nbeatsx')


In [ ]:
run_pipeline(str(config_path), split='test', start_from='backtest_all_models', stop_after='retrieve_nbeatsx')


In [ ]:
run_pipeline(str(config_path), split='test', start_from='evaluate_and_plot', stop_after='export_report_assets')


## Inspect outputs

In [ ]:
import pandas as pd
from IPython.display import display, Image

metrics_path = OUTPUT_ROOT / 'artifacts' / 'metrics' / 'test_metrics.csv'
display(pd.read_csv(metrics_path))
display(Image(filename=str(OUTPUT_ROOT / 'artifacts' / 'plots' / 'test_hourly_mae.png')))
display(Image(filename=str(OUTPUT_ROOT / 'artifacts' / 'plots' / 'test_high_vol_week.png')))
